Satrt point of checking dataset I will copy form exercise 2

In [1]:
import pandas as pd

df = pd.read_csv('dirty_cafe_sales.csv')
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [2]:
df.info()
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [3]:
df.isna().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

I can see the structure of this dataset and I can see the missing values. This time, to fill in the missing values, I will use the median where possible. The previous data was more of a reference guide, where it was better for the information to be missing than for it to be incorrect. In this case, since it is used for a report for a café, it is better to use the median simply so as not to spoil the overall data statistics.

In [5]:
df.dtypes

Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object

In [6]:
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
df.dtypes

Transaction ID              object
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object

I have noticed that sometimes in data I can see Quntity and Price Per Unit but no Total Price. I want to check, if have values for two field, can I fill the third filed

In [7]:
mask_qty = df['Quantity'].isna() & df['Price Per Unit'].notna() & df['Total Spent'].notna()
df.loc[mask_qty, 'Quantity'] = df.loc[mask_qty, 'Total Spent'] / df.loc[mask_qty, 'Price Per Unit']

mask_price = df['Price Per Unit'].isna() & df['Quantity'].notna() & df['Total Spent'].notna()
df.loc[mask_price, 'Price Per Unit'] = df.loc[mask_price, 'Total Spent'] / df.loc[mask_price, 'Quantity']

mask_total = df['Total Spent'].isna() & df['Quantity'].notna() & df['Price Per Unit'].notna()
df.loc[mask_total, 'Total Spent'] = df.loc[mask_total, 'Quantity'] * df.loc[mask_total, 'Price Per Unit']

df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Price Per Unit'].median())
df['Total Spent'] = df['Total Spent'].fillna(df['Total Spent'].median())


Remove the empty values

In [8]:
df.isna().sum()

Transaction ID         0
Item                 333
Quantity               0
Price Per Unit         0
Total Spent            0
Payment Method      2579
Location            3265
Transaction Date     460
dtype: int64

In [9]:
df['Item'] = df['Item'].fillna('Unknown')
df['Payment Method'] = df['Payment Method'].fillna('Unknown')
df['Location'] = df['Location'].fillna('Unknown')
df['Transaction Date'] = df['Transaction Date'].fillna('Unknown')
df.isna().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

Put the proper format for entryies 

In [10]:
for col in df.select_dtypes(include='object').columns:
    print(col, df[col].unique()[:10])

Transaction ID ['TXN_1961373' 'TXN_4977031' 'TXN_4271903' 'TXN_7034554' 'TXN_3160411'
 'TXN_2602893' 'TXN_4433211' 'TXN_6699534' 'TXN_4717867' 'TXN_2064365']
Item ['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'UNKNOWN' 'Sandwich'
 'Unknown' 'ERROR' 'Juice']
Payment Method ['Credit Card' 'Cash' 'UNKNOWN' 'Digital Wallet' 'ERROR' 'Unknown']
Location ['Takeaway' 'In-store' 'UNKNOWN' 'Unknown' 'ERROR']
Transaction Date [Timestamp('2023-09-08 00:00:00') Timestamp('2023-05-16 00:00:00')
 Timestamp('2023-07-19 00:00:00') Timestamp('2023-04-27 00:00:00')
 Timestamp('2023-06-11 00:00:00') Timestamp('2023-03-31 00:00:00')
 Timestamp('2023-10-06 00:00:00') Timestamp('2023-10-28 00:00:00')
 Timestamp('2023-07-28 00:00:00') Timestamp('2023-12-31 00:00:00')]


In [11]:
df['Item'] = df['Item'].str.strip().str.title()
df['Payment Method'] = df['Payment Method'].str.strip().str.title()
df['Location'] = df['Location'].str.strip().str.title()
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce').dt.strftime('%Y-%m-%d')
df['Transaction Date'] = df['Transaction Date'].fillna('Unknown')
for col in df.select_dtypes(include='object').columns:
    print(col, df[col].unique()[:10])

Transaction ID ['TXN_1961373' 'TXN_4977031' 'TXN_4271903' 'TXN_7034554' 'TXN_3160411'
 'TXN_2602893' 'TXN_4433211' 'TXN_6699534' 'TXN_4717867' 'TXN_2064365']
Item ['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'Unknown' 'Sandwich' 'Error'
 'Juice' 'Tea']
Payment Method ['Credit Card' 'Cash' 'Unknown' 'Digital Wallet' 'Error']
Location ['Takeaway' 'In-Store' 'Unknown' 'Error']
Transaction Date ['2023-09-08' '2023-05-16' '2023-07-19' '2023-04-27' '2023-06-11'
 '2023-03-31' '2023-10-06' '2023-10-28' '2023-07-28' '2023-12-31']


Now I will check numeric field on some mistakes in values

In [12]:
df[df['Quantity'] < 0]
df[df['Price Per Unit'] < 0]
df[df['Total Spent'] < 0]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date


In [13]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    10000 non-null  object 
 1   Item              10000 non-null  object 
 2   Quantity          10000 non-null  float64
 3   Price Per Unit    10000 non-null  float64
 4   Total Spent       10000 non-null  float64
 5   Payment Method    10000 non-null  object 
 6   Location          10000 non-null  object 
 7   Transaction Date  10000 non-null  object 
dtypes: float64(3), object(5)
memory usage: 625.1+ KB


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-Store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-Store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-Store,2023-06-11


In [14]:
df.describe(include='all')

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,10000,10000.00000,10000.000000,10000.000000,10000,10000,10000
unique,10000,10,NaN,NaN,NaN,5,4,366
top,TXN_1961373,Juice,NaN,NaN,NaN,Unknown,Unknown,Unknown
freq,1,1171,NaN,NaN,NaN,2872,3603,460
mean,NaN,NaN,3.02550,2.948100,8.927200,NaN,NaN,NaN
std,NaN,NaN,1.41748,1.277329,5.992741,NaN,NaN,NaN
min,NaN,NaN,1.00000,1.000000,1.000000,NaN,NaN,NaN
25%,NaN,NaN,2.00000,2.000000,4.000000,NaN,NaN,NaN
50%,NaN,NaN,3.00000,3.000000,8.000000,NaN,NaN,NaN
75%,NaN,NaN,4.00000,4.000000,12.000000,NaN,NaN,NaN


In [ ]:
df.to_csv("clean_cafe_sales.csv", index=False)